# A2 — DictaBERT Morphological & NER Parsing

In [ ]:
import os
import json
import time
from pathlib import Path
from transformers import AutoTokenizer, AutoModel

BASE_DIR = "/Users/I771657/Library/CloudStorage/OneDrive-SAPSE/Documents/School/MINI"
CORPUS_PATH = os.path.join(BASE_DIR, "data", "corpus.json")
PARSED_DIR = os.path.join(BASE_DIR, "data", "parsed")
Path(PARSED_DIR).mkdir(parents=True, exist_ok=True)

with open(CORPUS_PATH, encoding="utf-8") as f:
    corpus = json.load(f)
print(f"Loaded {len(corpus)} songs")

In [ ]:
print("Loading dicta-il/dictabert-tiny-joint...")
tokenizer = AutoTokenizer.from_pretrained('dicta-il/dictabert-tiny-joint', trust_remote_code=True)
morph_model = AutoModel.from_pretrained('dicta-il/dictabert-tiny-joint', trust_remote_code=True)
morph_model.eval()
print("Model loaded.")

In [ ]:
def extract_morphology(result_dict: dict) -> list:
    """Extract token-level morphological data. result_dict is result[0]."""
    tokens = []
    for token_data in result_dict.get("tokens", []):
        morph = token_data.get("morph", {})
        syntax = token_data.get("syntax", {})
        tokens.append({
            "word": token_data.get("token", ""),
            "lemma": token_data.get("lex", token_data.get("token", "")),
            "pos": morph.get("pos", ""),
            "dep": syntax.get("dep_func", ""),
            "dep_head": syntax.get("dep_head", ""),
            "feats": morph.get("feats", {}),
        })
    return tokens


def extract_ner(result_dict: dict) -> list:
    """Extract NER entities from the ner_entities field of the joint model output."""
    entities = []
    for entity in result_dict.get("ner_entities", []):
        entities.append({
            "word": entity.get("phrase", ""),
            "label": entity.get("label", ""),
            "start": entity.get("start", 0),
            "end": entity.get("end", 0),
        })
    return entities

In [ ]:
sample = corpus[0]
print(f"Spot-check: song {sample['song_id']} — {sample['title']}")
raw = morph_model.predict([sample["text"]], tokenizer, output_style='json')
morph = extract_morphology(raw[0])
ner = extract_ner(raw[0])
print(f"Tokens: {len(morph)}")
print("First 5 tokens:")
for t in morph[:5]:
    print(f"  {t['word']:15s}  POS={t['pos']:8s}  lemma={t['lemma']:15s}  dep={t['dep']}")
print(f"NER entities: {ner}")

In [ ]:
print("Parsing all songs...")
start = time.time()

for i, record in enumerate(corpus):
    song_id = record["song_id"]
    out_path = os.path.join(PARSED_DIR, f"{song_id:03d}.json")

    if os.path.exists(out_path):
        continue

    raw = morph_model.predict([record["text"]], tokenizer, output_style='json')

    parsed = {
        "song_id": song_id,
        "title": record["title"],
        "year": record["year"],
        "album": record["album"],
        "morphology": extract_morphology(raw[0]),
        "ner": extract_ner(raw[0]),
    }

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(parsed, f, ensure_ascii=False, indent=2)

    if (i + 1) % 10 == 0:
        print(f"  {i + 1}/167 songs ({time.time() - start:.0f}s elapsed)")

n_files = len(list(Path(PARSED_DIR).glob("*.json")))
print(f"\nDone. {n_files} files in data/parsed/ ({time.time() - start:.0f}s total)")

In [ ]:
# Summary stats across all parsed files
total_tokens = 0
total_ner = 0
pos_counts = {}

for record in corpus:
    path = os.path.join(PARSED_DIR, f"{record['song_id']:03d}.json")
    with open(path, encoding="utf-8") as f:
        p = json.load(f)
    total_tokens += len(p["morphology"])
    total_ner += len(p["ner"])
    for t in p["morphology"]:
        pos = t["pos"]
        pos_counts[pos] = pos_counts.get(pos, 0) + 1

print(f"Total tokens across corpus: {total_tokens}")
print(f"Total NER entities: {total_ner}")
print(f"Avg tokens per song: {total_tokens / len(corpus):.0f}")
print("\nPOS distribution:")
for pos, count in sorted(pos_counts.items(), key=lambda x: -x[1]):
    pct = 100 * count / total_tokens
    print(f"  {pos:10s} {count:6d}  ({pct:.1f}%)")